# Notebook 09: Head-to-Head: Comparing Alignment Methods

**Portfolio Notebook** -- Produces publishable-quality comparison results across DPO, KTO, and SimPO.

**Runtime**: ~45-60 min on Colab T4 (all three training runs + evaluation)  
**GPU Memory**: ~8 GB peak  
**Objective**: Rigorously compare alignment methods under fair experimental conditions and produce a clear visual dashboard of results.

---
## 1. Self-Quiz: Active Recall (Answer Before Reading On)

Try answering these from memory before proceeding:

1. **Which alignment method is "best"?** Is there even a universal answer to this?
2. **What metrics should we use to compare alignment methods?** List at least 5.
3. **What makes a fair comparison?** What variables must be controlled?
4. **DPO vs KTO**: What is the fundamental data requirement difference?
5. **SimPO vs DPO**: What architectural simplification does SimPO introduce?
6. **Can you compare offline methods (DPO) with online methods (PPO-RLHF) fairly?** Why or why not?
7. **What is the typical failure mode when comparing methods with different hyperparameter budgets?**

<details>
<summary>Click to reveal key points</summary>

1. No universal best -- depends on data quality, compute budget, and task domain. DPO often wins on efficiency, RLHF on ceiling performance.
2. Reward model score, win rate (LLM-as-judge), KL divergence from reference, response length distribution, perplexity, diversity, toxicity, factuality.
3. Same base model, same data, same compute budget (wall-clock or FLOPs), same hyperparameter search budget, same evaluation protocol.
4. DPO needs paired preferences (chosen vs rejected for same prompt). KTO only needs binary labels (good/bad) -- unpaired.
5. SimPO removes the reference model entirely, using sequence-level likelihood as the implicit reward.
6. Hard to compare fairly because online methods see model-generated data during training. Matched compute comparisons are most informative.
7. The method with more hyperparameter tuning appears better, but the advantage is from tuning, not the method itself.
</details>

---
## 2. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets trl accelerate peft matplotlib pandas seaborn scikit-learn

In [ ]:
import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
)
from datasets import load_dataset, Dataset
from trl import (
    SFTTrainer,
    SFTConfig,
    DPOTrainer,
    DPOConfig,
    KTOTrainer,
    KTOConfig,
    CPOTrainer,
    CPOConfig,
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import seaborn as sns
import numpy as np
from collections import defaultdict
import json
import time
import warnings
import os
import gc

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="colorblind")

# Detect hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Use RunPod model if enough VRAM, otherwise GPT-2
if device == "cuda" and torch.cuda.get_device_properties(0).total_memory > 12e9:
    BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    print("Using TinyLlama-1.1B (RunPod mode)")
else:
    BASE_MODEL = "gpt2"
    print("Using GPT-2 (Colab mode)")

---
## 3. Experimental Design

### Why Fair Comparisons Are Hard

Most published comparisons of alignment methods are **not fair**. Common pitfalls:

| Pitfall | Why It Matters |
|---------|---------------|
| Different base models | Confounds method quality with model quality |
| Different training data | A better dataset makes any method look good |
| Different compute budgets | More gradient steps = better results for any method |
| Different hyperparameter search | The method that got tuned more wins |
| Cherry-picked examples | Qualitative examples can be misleading |
| Single evaluation metric | No single metric captures alignment quality |

### Our Experimental Protocol

We control for all of these:

- **Base model**: Same SFT checkpoint for all methods
- **Data**: Same preference data, reformatted per method's requirements
- **Compute**: Same number of gradient steps, same batch size
- **Hyperparameters**: Default or lightly tuned, documented
- **Evaluation**: Multiple metrics, both quantitative and qualitative

### Methods Under Comparison

| Method | Data Format | Reference Model | Key Hyperparameter |
|--------|------------|-----------------|--------------------|
| DPO | Paired preferences | Yes (frozen) | beta (KL weight) |
| KTO | Unpaired binary | Yes (frozen) | beta, desirable_weight |
| SimPO | Paired preferences | No | beta, gamma (margin) |

### Metrics

1. **Reward model score**: Proxy for alignment quality
2. **Win rate vs SFT**: How much better than the baseline?
3. **KL divergence from reference**: How far did we deviate?
4. **Response length**: Are we gaming length?
5. **Perplexity**: Is the model still fluent?
6. **Training wall-clock time**: Practical efficiency

---
## 4. Data Pipeline

In [ ]:
# Load Anthropic HH-RLHF dataset
# This contains paired human preferences: chosen vs rejected responses
print("Loading Anthropic HH-RLHF dataset...")
raw_dataset = load_dataset("Anthropic/hh-rlhf", split="train")
print(f"Total examples: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nExample:")
print(f"  Chosen (first 200 chars): {raw_dataset[0]['chosen'][:200]}")
print(f"  Rejected (first 200 chars): {raw_dataset[0]['rejected'][:200]}")

In [ ]:
def parse_hh_conversation(text):
    """Parse Anthropic HH-RLHF format into prompt and response.
    
    The HH-RLHF format uses \n\nHuman: and \n\nAssistant: markers.
    We extract the last assistant turn as the response, everything before as prompt.
    """
    # Split on the last Assistant turn
    parts = text.rsplit("\n\nAssistant:", 1)
    if len(parts) == 2:
        prompt = parts[0] + "\n\nAssistant:"
        response = parts[1].strip()
    else:
        prompt = text
        response = ""
    return prompt, response


def prepare_preference_data(dataset, max_samples=2000, max_length=256):
    """Convert HH-RLHF to the format needed for alignment training.
    
    Returns:
        paired_data: For DPO/SimPO -- (prompt, chosen, rejected) triples
        unpaired_data: For KTO -- (prompt, completion, label) where label is True/False
    """
    paired_records = []
    unpaired_records = []
    
    for i, example in enumerate(dataset):
        if i >= max_samples:
            break
            
        chosen_prompt, chosen_response = parse_hh_conversation(example["chosen"])
        rejected_prompt, rejected_response = parse_hh_conversation(example["rejected"])
        
        # Skip if prompts don't match (data quality check)
        if chosen_prompt != rejected_prompt:
            # Use the chosen prompt as canonical
            pass
        
        # Skip very short or very long responses
        if len(chosen_response) < 10 or len(rejected_response) < 10:
            continue
        if len(chosen_response) > max_length * 4 or len(rejected_response) > max_length * 4:
            continue
            
        # Paired format for DPO/SimPO
        paired_records.append({
            "prompt": chosen_prompt,
            "chosen": chosen_response,
            "rejected": rejected_response,
        })
        
        # Unpaired format for KTO: split each pair into two independent examples
        unpaired_records.append({
            "prompt": chosen_prompt,
            "completion": chosen_response,
            "label": True,  # desirable
        })
        unpaired_records.append({
            "prompt": chosen_prompt,  # Use same prompt for rejected too
            "completion": rejected_response,
            "label": False,  # undesirable
        })
    
    paired_dataset = Dataset.from_list(paired_records)
    unpaired_dataset = Dataset.from_list(unpaired_records)
    
    return paired_dataset, unpaired_dataset


# Prepare data
N_TRAIN = 1500
N_EVAL = 300

paired_data, unpaired_data = prepare_preference_data(raw_dataset, max_samples=N_TRAIN + N_EVAL)

# Split into train and eval
paired_split = paired_data.train_test_split(test_size=N_EVAL / len(paired_data), seed=42)
paired_train = paired_split["train"]
paired_eval = paired_split["test"]

# For KTO: NOTE -- a random split does NOT keep the two examples derived from the
# same preference pair together; one prompt's chosen example can land in train while
# its rejected example lands in eval (prompt-level leakage). For a clean split,
# group by prompt first, e.g.:
#   unique_prompts = list(dict.fromkeys(unpaired_data["prompt"]))
#   eval_prompt_set = set(unique_prompts[:N_EVAL])  # or a seeded random subset
#   unpaired_train = unpaired_data.filter(lambda x: x["prompt"] not in eval_prompt_set)
#   unpaired_eval  = unpaired_data.filter(lambda x: x["prompt"] in eval_prompt_set)
# We keep the simple random split here for demo purposes.
unpaired_split = unpaired_data.train_test_split(test_size=N_EVAL * 2 / len(unpaired_data), seed=42)
unpaired_train = unpaired_split["train"]
unpaired_eval = unpaired_split["test"]

print(f"Paired train: {len(paired_train)}, eval: {len(paired_eval)}")
print(f"Unpaired train: {len(unpaired_train)}, eval: {len(unpaired_eval)}")
print(f"\nSample paired example:")
print(f"  Prompt: {paired_train[0]['prompt'][:100]}...")
print(f"  Chosen: {paired_train[0]['chosen'][:100]}...")
print(f"  Rejected: {paired_train[0]['rejected'][:100]}...")

In [ ]:
# Also prepare SFT data: just the chosen responses
def prepare_sft_data(paired_dataset):
    """Create SFT dataset from chosen responses only."""
    records = []
    for example in paired_dataset:
        text = example["prompt"] + " " + example["chosen"]
        records.append({"text": text})
    return Dataset.from_list(records)

sft_train = prepare_sft_data(paired_train)
sft_eval = prepare_sft_data(paired_eval)
print(f"SFT train: {len(sft_train)}, eval: {len(sft_eval)}")

---
## 5. Train All Methods

We train in order:
1. SFT baseline (shared starting point for all methods)
2. DPO
3. KTO
4. SimPO

In [ ]:
# Load base model and tokenizer
print(f"Loading base model: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Common training parameters -- SAME for all methods for fairness
MAX_SEQ_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch size = 16
LEARNING_RATE = 5e-5
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = 200  # Fixed compute budget
WARMUP_STEPS = 20
LOGGING_STEPS = 10
OUTPUT_DIR_BASE = "./comparison_checkpoints"

os.makedirs(OUTPUT_DIR_BASE, exist_ok=True)

# Dictionary to store training logs for each method
training_logs = {}
training_times = {}

### 5a. SFT Baseline

In [ ]:
# Step 1: SFT baseline -- the shared starting point
print("=" * 60)
print("Training SFT Baseline")
print("=" * 60)

sft_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
sft_model.config.pad_token_id = tokenizer.pad_token_id

sft_config = SFTConfig(
    output_dir=f"{OUTPUT_DIR_BASE}/sft",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",
    fp16=device == "cuda",
    dataset_text_field="text",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    processing_class=tokenizer,
)

start_time = time.time()
sft_result = sft_trainer.train()
training_times["SFT"] = time.time() - start_time
training_logs["SFT"] = sft_trainer.state.log_history

# Save SFT model -- this becomes the reference for all alignment methods
sft_model.save_pretrained(f"{OUTPUT_DIR_BASE}/sft")
tokenizer.save_pretrained(f"{OUTPUT_DIR_BASE}/sft")
print(f"\nSFT training completed in {training_times['SFT']:.1f}s")
print(f"Final loss: {sft_result.training_loss:.4f}")

# Clear GPU memory
del sft_trainer
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

### 5b. DPO Training

In [ ]:
# Step 2: DPO training
print("=" * 60)
print("Training DPO")
print("=" * 60)

# Load fresh copy from SFT checkpoint
dpo_model = AutoModelForCausalLM.from_pretrained(f"{OUTPUT_DIR_BASE}/sft")
ref_model = AutoModelForCausalLM.from_pretrained(f"{OUTPUT_DIR_BASE}/sft")

dpo_config = DPOConfig(
    output_dir=f"{OUTPUT_DIR_BASE}/dpo",
    beta=0.1,  # KL penalty weight -- standard default
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",
    fp16=device == "cuda",
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=paired_train,
    eval_dataset=paired_eval,
    processing_class=tokenizer,
)

start_time = time.time()
dpo_result = dpo_trainer.train()
training_times["DPO"] = time.time() - start_time
training_logs["DPO"] = dpo_trainer.state.log_history

dpo_model.save_pretrained(f"{OUTPUT_DIR_BASE}/dpo")
print(f"\nDPO training completed in {training_times['DPO']:.1f}s")
print(f"Final loss: {dpo_result.training_loss:.4f}")

del dpo_trainer, ref_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

### 5c. KTO Training

In [ ]:
# Step 3: KTO training
print("=" * 60)
print("Training KTO")
print("=" * 60)

kto_model = AutoModelForCausalLM.from_pretrained(f"{OUTPUT_DIR_BASE}/sft")
ref_model_kto = AutoModelForCausalLM.from_pretrained(f"{OUTPUT_DIR_BASE}/sft")

kto_config = KTOConfig(
    output_dir=f"{OUTPUT_DIR_BASE}/kto",
    beta=0.1,
    desirable_weight=1.0,
    undesirable_weight=1.0,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",
    fp16=device == "cuda",
    remove_unused_columns=False,
)

kto_trainer = KTOTrainer(
    model=kto_model,
    ref_model=ref_model_kto,
    args=kto_config,
    train_dataset=unpaired_train,
    eval_dataset=unpaired_eval,
    processing_class=tokenizer,
)

start_time = time.time()
kto_result = kto_trainer.train()
training_times["KTO"] = time.time() - start_time
training_logs["KTO"] = kto_trainer.state.log_history

kto_model.save_pretrained(f"{OUTPUT_DIR_BASE}/kto")
print(f"\nKTO training completed in {training_times['KTO']:.1f}s")
print(f"Final loss: {kto_result.training_loss:.4f}")

del kto_trainer, ref_model_kto
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

### 5d. SimPO Training

SimPO (Simple Preference Optimization) removes the reference model, using length-normalized average log-probability as the implicit reward. In TRL, SimPO is implemented inside **CPOTrainer** (not DPOTrainer): use `CPOConfig(loss_type="simpo", cpo_alpha=0.0, simpo_gamma=...)`. Note that `DPOTrainer(ref_model=None)` does **not** give you a reference-free method -- DPOTrainer silently creates a reference copy of the model.

> **Version note**: TRL reached v1.x in 2026 and trainer APIs have moved between releases more than once -- verify this cell against the docs for your installed TRL version before running.

In [ ]:
# Step 4: SimPO training
# IMPORTANT: in TRL, SimPO is NOT a DPOTrainer loss type. It is implemented in
# CPOTrainer via CPOConfig(loss_type="simpo", cpo_alpha=0.0, simpo_gamma=...).
# (Passing ref_model=None to DPOTrainer does NOT make it reference-free: the
# trainer silently creates a reference copy of the model.)
# TRL reached v1.x in 2026 and APIs have moved between releases -- verify
# against your installed TRL version before running.
print("=" * 60)
print("Training SimPO")
print("=" * 60)

simpo_model = AutoModelForCausalLM.from_pretrained(f"{OUTPUT_DIR_BASE}/sft")

# SimPO: reference-free, uses length-normalized average log likelihood as reward
simpo_config = CPOConfig(
    output_dir=f"{OUTPUT_DIR_BASE}/simpo",
    beta=2.0,  # SimPO typically uses larger beta since no explicit KL
    loss_type="simpo",  # SimPO variant of the CPO loss
    cpo_alpha=0.0,  # 0.0 = pure SimPO (no behavior-cloning/NLL regularizer)
    simpo_gamma=0.5,  # target reward margin gamma
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",
    fp16=device == "cuda",
    remove_unused_columns=False,
)

simpo_trainer = CPOTrainer(
    model=simpo_model,  # reference-free: no ref_model argument at all
    args=simpo_config,
    train_dataset=paired_train,
    eval_dataset=paired_eval,
    processing_class=tokenizer,
)

start_time = time.time()
simpo_result = simpo_trainer.train()
training_times["SimPO"] = time.time() - start_time
training_logs["SimPO"] = simpo_trainer.state.log_history

simpo_model.save_pretrained(f"{OUTPUT_DIR_BASE}/simpo")
print(f"\nSimPO training completed in {training_times['SimPO']:.1f}s")
print(f"Final loss: {simpo_result.training_loss:.4f}")

del simpo_trainer
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("All training runs complete!")
print("=" * 60)
for method, t in training_times.items():
    print(f"  {method}: {t:.1f}s ({t/60:.1f} min)")

---
## 6. Quantitative Evaluation

In [ ]:
# Load all models for evaluation
print("Loading models for evaluation...")
models = {}
model_dirs = {
    "SFT": f"{OUTPUT_DIR_BASE}/sft",
    "DPO": f"{OUTPUT_DIR_BASE}/dpo",
    "KTO": f"{OUTPUT_DIR_BASE}/kto",
    "SimPO": f"{OUTPUT_DIR_BASE}/simpo",
}

for name, path in model_dirs.items():
    models[name] = AutoModelForCausalLM.from_pretrained(path).to(device)
    models[name].eval()
    print(f"  Loaded {name}")

# Reference model for KL computation
ref_model_eval = AutoModelForCausalLM.from_pretrained(f"{OUTPUT_DIR_BASE}/sft").to(device)
ref_model_eval.eval()
print("  Loaded reference model")

In [ ]:
@torch.no_grad()
def generate_response(model, tokenizer, prompt, max_new_tokens=128):
    """Generate a response from a model given a prompt."""
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH // 2
    ).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


@torch.no_grad()
def compute_log_probs(model, tokenizer, text, max_length=MAX_SEQ_LENGTH):
    """Compute per-token log probabilities for a given text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    outputs = model(**inputs)
    logits = outputs.logits[:, :-1, :]  # shift right
    labels = inputs.input_ids[:, 1:]  # shift left
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    return token_log_probs


@torch.no_grad()
def compute_kl_divergence(model, ref_model, tokenizer, text):
    """Compute KL(model || ref) for a given text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)
    
    model_outputs = model(**inputs)
    ref_outputs = ref_model(**inputs)
    
    model_logprobs = F.log_softmax(model_outputs.logits, dim=-1)
    ref_logprobs = F.log_softmax(ref_outputs.logits, dim=-1)
    
    # KL(model || ref) = sum model_prob * (log model_prob - log ref_prob)
    model_probs = model_logprobs.exp()
    kl = (model_probs * (model_logprobs - ref_logprobs)).sum(dim=-1).mean()
    return kl.item()


@torch.no_grad()
def compute_perplexity(model, tokenizer, text):
    """Compute perplexity of text under model."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)
    outputs = model(**inputs, labels=inputs.input_ids)
    return torch.exp(outputs.loss).item()


print("Evaluation functions defined.")

In [ ]:
# Generate responses from all models on evaluation prompts
N_EVAL_PROMPTS = 50  # Number of prompts to evaluate on
eval_prompts = [paired_eval[i]["prompt"] for i in range(min(N_EVAL_PROMPTS, len(paired_eval)))]

print(f"Generating responses from all models on {len(eval_prompts)} prompts...")
all_responses = {name: [] for name in models}

for i, prompt in enumerate(eval_prompts):
    if i % 10 == 0:
        print(f"  Prompt {i+1}/{len(eval_prompts)}...")
    for name, model in models.items():
        response = generate_response(model, tokenizer, prompt)
        all_responses[name].append(response)

print("Generation complete.")

In [ ]:
# Compute all metrics
print("Computing metrics...")

metrics = {name: {"kl_div": [], "perplexity": [], "response_length": [], "self_confidence": []} 
           for name in models}

for i, prompt in enumerate(eval_prompts):
    if i % 10 == 0:
        print(f"  Evaluating prompt {i+1}/{len(eval_prompts)}...")
    
    for name, model in models.items():
        response = all_responses[name][i]
        full_text = prompt + " " + response
        
        # Response length (in tokens)
        resp_tokens = tokenizer(response, return_tensors="pt").input_ids.shape[1]
        metrics[name]["response_length"].append(resp_tokens)
        
        # KL divergence from reference
        if name != "SFT":  # SFT IS the reference
            kl = compute_kl_divergence(model, ref_model_eval, tokenizer, full_text)
            metrics[name]["kl_div"].append(kl)
        else:
            metrics[name]["kl_div"].append(0.0)
        
        # Perplexity (using reference model as the scorer)
        ppl = compute_perplexity(ref_model_eval, tokenizer, full_text)
        metrics[name]["perplexity"].append(min(ppl, 1000))  # cap extreme values
        
        # Fluency/self-confidence proxy: each model's negative perplexity on ITS OWN
        # generation. CAVEAT: this is NOT a reward proxy. A model is by construction
        # confident in its own samples, and degenerate/repetitive outputs get HIGHER
        # scores. For a real reward proxy, score all generations under ONE fixed
        # scorer model (or a trained reward model / LLM-as-judge).
        own_ppl = compute_perplexity(model, tokenizer, full_text)
        metrics[name]["self_confidence"].append(-min(own_ppl, 1000))

print("Metrics computation complete.")

In [ ]:
# Compute win rates: each method vs SFT baseline
# Using a simple heuristic: lower perplexity under reference = better
# In a real experiment, you would use an LLM-as-judge or a reward model

def compute_win_rate(method_responses, sft_responses, eval_prompts, ref_model, tokenizer):
    """Compute win rate of method vs SFT using reference model perplexity as proxy.
    
    A response 'wins' if it achieves lower perplexity under the reference model,
    suggesting it is more fluent and coherent.
    
    In production, you would use:
    - A trained reward model (e.g., from OpenAssistant)
    - LLM-as-judge (GPT-4 or Claude rating both responses)
    - MT-Bench style scoring
    """
    wins = 0
    ties = 0
    total = len(eval_prompts)
    
    for i in range(total):
        method_text = eval_prompts[i] + " " + method_responses[i]
        sft_text = eval_prompts[i] + " " + sft_responses[i]
        
        method_ppl = compute_perplexity(ref_model, tokenizer, method_text)
        sft_ppl = compute_perplexity(ref_model, tokenizer, sft_text)
        
        # Also factor in response length -- penalize very short or very long
        method_len = len(method_responses[i].split())
        sft_len = len(sft_responses[i].split())
        
        # Combined score: lower perplexity + reasonable length
        # CAVEAT: penalizing deviation from ~50 words directly confounds the
        # length-exploitation analysis -- any method that systematically changes
        # response length is penalized here regardless of actual quality, so do
        # not read the win rates as evidence about length effects.
        method_score = -method_ppl - 0.1 * abs(method_len - 50)  # penalize deviation from ~50 words
        sft_score = -sft_ppl - 0.1 * abs(sft_len - 50)
        
        if method_score > sft_score + 0.5:
            wins += 1
        elif abs(method_score - sft_score) <= 0.5:
            ties += 1
    
    win_rate = wins / total
    tie_rate = ties / total
    return win_rate, tie_rate


win_rates = {}
tie_rates = {}
for name in ["DPO", "KTO", "SimPO"]:
    wr, tr = compute_win_rate(
        all_responses[name], all_responses["SFT"], eval_prompts, ref_model_eval, tokenizer
    )
    win_rates[name] = wr
    tie_rates[name] = tr
    print(f"{name} vs SFT: win={wr:.1%}, tie={tr:.1%}, loss={1-wr-tr:.1%}")

In [ ]:
# Compile all metrics into a summary table
summary_data = []
for name in models:
    row = {
        "Method": name,
        "Avg KL Divergence": np.mean(metrics[name]["kl_div"]),
        "Avg Perplexity": np.mean(metrics[name]["perplexity"]),
        "Avg Response Length": np.mean(metrics[name]["response_length"]),
        "Self-Confidence (neg own-ppl)": np.mean(metrics[name]["self_confidence"]),
        "Win Rate vs SFT": win_rates.get(name, "--"),
        "Training Time (s)": training_times.get(name, 0),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.set_index("Method")
print("\n" + "=" * 80)
print("COMPREHENSIVE METRICS COMPARISON")
print("=" * 80)
print(summary_df.to_string())
print("\n(Lower KL = closer to reference; Lower perplexity = more fluent)")
print("(Higher win rate = better than SFT baseline)")
print("(Self-Confidence = each model's neg. perplexity on its OWN generation;")
print(" NOT a reward proxy -- it can reward degenerate, repetitive outputs)")

---
## 7. Visualization Dashboard

In [ ]:
# Color scheme for consistency across all plots
METHOD_COLORS = {
    "SFT": "#7f7f7f",   # gray (baseline)
    "DPO": "#1f77b4",   # blue
    "KTO": "#ff7f0e",   # orange
    "SimPO": "#2ca02c", # green
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Alignment Methods Comparison Dashboard", fontsize=16, fontweight="bold", y=1.02)

# ---- Plot 1: Win Rates ----
ax = axes[0, 0]
methods = ["DPO", "KTO", "SimPO"]
wr_values = [win_rates[m] for m in methods]
colors = [METHOD_COLORS[m] for m in methods]
bars = ax.bar(methods, wr_values, color=colors, edgecolor="black", linewidth=0.8)
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.7, label="50% (no improvement)")
ax.set_ylabel("Win Rate vs SFT")
ax.set_title("Win Rates Against SFT Baseline")
ax.set_ylim(0, 1.0)
ax.legend()
for bar, val in zip(bars, wr_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f"{val:.1%}", ha="center", fontweight="bold")

# ---- Plot 2: Training Loss Curves ----
ax = axes[0, 1]
for name in ["SFT", "DPO", "KTO", "SimPO"]:
    logs = training_logs[name]
    steps = [l["step"] for l in logs if "loss" in l]
    losses = [l["loss"] for l in logs if "loss" in l]
    if steps and losses:
        ax.plot(steps, losses, label=name, color=METHOD_COLORS[name], linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Curves")
ax.legend()
ax.set_yscale("log")

# ---- Plot 3: KL Divergence ----
ax = axes[0, 2]
kl_methods = ["DPO", "KTO", "SimPO"]
kl_means = [np.mean(metrics[m]["kl_div"]) for m in kl_methods]
kl_stds = [np.std(metrics[m]["kl_div"]) for m in kl_methods]
colors = [METHOD_COLORS[m] for m in kl_methods]
bars = ax.bar(kl_methods, kl_means, yerr=kl_stds, color=colors, 
              edgecolor="black", linewidth=0.8, capsize=5)
ax.set_ylabel("KL Divergence from Reference")
ax.set_title("KL Divergence (lower = closer to SFT)")

# ---- Plot 4: Self-Confidence Distributions (Box Plot) ----
# NOTE: this is each model's neg. perplexity on its OWN generations -- a
# fluency/self-confidence proxy, NOT a reward proxy (it can reward degeneration).
ax = axes[1, 0]
reward_data = []
for name in models:
    for val in metrics[name]["self_confidence"]:
        reward_data.append({"Method": name, "Self-Confidence": val})
reward_df = pd.DataFrame(reward_data)
box_order = ["SFT", "DPO", "KTO", "SimPO"]
bp = sns.boxplot(data=reward_df, x="Method", y="Self-Confidence", order=box_order,
                 palette=METHOD_COLORS, ax=ax)
ax.set_title("Self-Confidence Distributions (NOT a reward proxy)")
ax.set_ylabel("Neg. own-perplexity (fluency/self-confidence)")

# ---- Plot 5: Response Length Violin Plots ----
ax = axes[1, 1]
length_data = []
for name in models:
    for val in metrics[name]["response_length"]:
        length_data.append({"Method": name, "Response Length (tokens)": val})
length_df = pd.DataFrame(length_data)
sns.violinplot(data=length_df, x="Method", y="Response Length (tokens)", order=box_order,
               palette=METHOD_COLORS, ax=ax, inner="box")
ax.set_title("Response Length Distributions")

# ---- Plot 6: Quality vs Compute Cost Scatter ----
ax = axes[1, 2]
for name in ["DPO", "KTO", "SimPO"]:
    ax.scatter(
        training_times[name],
        win_rates[name],
        s=200,
        c=METHOD_COLORS[name],
        edgecolors="black",
        linewidth=1.5,
        label=name,
        zorder=5,
    )
    ax.annotate(name, (training_times[name], win_rates[name]),
                textcoords="offset points", xytext=(10, 5), fontsize=11, fontweight="bold")
ax.set_xlabel("Training Time (seconds)")
ax.set_ylabel("Win Rate vs SFT")
ax.set_title("Quality vs Compute Cost")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("alignment_comparison_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard saved to alignment_comparison_dashboard.png")

In [ ]:
# Publication-quality summary table
print("\n" + "=" * 90)
print("TABLE: Alignment Methods Comparison (all methods trained from same SFT checkpoint)")
print("=" * 90)

table_data = []
for name in ["SFT", "DPO", "KTO", "SimPO"]:
    row = {
        "Method": name,
        "Win Rate": f"{win_rates.get(name, 0.5):.1%}",
        "KL Div.": f"{np.mean(metrics[name]['kl_div']):.3f}",
        "Perplexity": f"{np.mean(metrics[name]['perplexity']):.1f}",
        "Avg Length": f"{np.mean(metrics[name]['response_length']):.0f}",
        "Time (s)": f"{training_times.get(name, 0):.0f}",
        "Ref Model?": "N/A" if name == "SFT" else ("No" if name == "SimPO" else "Yes"),
    }
    table_data.append(row)

table_df = pd.DataFrame(table_data).set_index("Method")
print(table_df.to_string())
print("\nNote: Win rates computed against SFT baseline using perplexity-based proxy.")
print("For publication, use a trained reward model or LLM-as-judge.")

---
## 8. Qualitative Examples

We select 5 diverse prompts and show all models' responses side by side.

In [ ]:
# Select 5 diverse evaluation prompts
# Try to pick prompts that test different capabilities
diverse_indices = [0, len(eval_prompts)//5, 2*len(eval_prompts)//5, 
                   3*len(eval_prompts)//5, 4*len(eval_prompts)//5]
diverse_indices = [min(i, len(eval_prompts)-1) for i in diverse_indices]

for idx, i in enumerate(diverse_indices):
    prompt = eval_prompts[i]
    print("\n" + "=" * 80)
    print(f"EXAMPLE {idx + 1}")
    print("=" * 80)
    print(f"PROMPT: {prompt[:300]}...")
    print("-" * 80)
    
    for name in ["SFT", "DPO", "KTO", "SimPO"]:
        response = all_responses[name][i]
        # Truncate for display
        display_response = response[:300] + ("..." if len(response) > 300 else "")
        print(f"\n[{name}]: {display_response}")
    
    print()

In [ ]:
# Annotated comparison on a single example
print("\n" + "=" * 80)
print("DETAILED ANNOTATION: Example 1")
print("=" * 80)

prompt = eval_prompts[0]
print(f"PROMPT: {prompt[:400]}")
print()

for name in ["SFT", "DPO", "KTO", "SimPO"]:
    response = all_responses[name][0]
    resp_len = len(tokenizer(response).input_ids)
    print(f"--- {name} ({resp_len} tokens) ---")
    print(response[:500])
    print()

print("ANALYSIS:")
print("- SFT: Baseline response. Typically generic, may not fully address the query.")
print("- DPO: Should show improved adherence to user preferences. Watch for helpfulness.")
print("- KTO: Trained on individual quality signals. May differ subtly from DPO.")
print("- SimPO: No reference model constraint. May show more diverse or more extreme outputs.")
print("\nKey question: Do the alignment methods produce qualitatively different responses,")
print("or are the differences subtle (as is common with small models and limited data)?")

---
## 9. Analysis and Conclusions

### Key Findings

Based on our controlled comparison:

**1. Which method won?**
- At this scale (GPT-2 / TinyLlama, ~1.5K examples, 200 steps), differences between methods are often **modest**.
- This itself is an important finding: **method choice matters less than data quality and compute at small scale**.
- At larger scale (7B+ models, 100K+ examples), DPO and SimPO typically perform comparably, with KTO slightly behind on paired-preference benchmarks.

**2. When would you choose each method?**

| Scenario | Best Method | Why |
|----------|------------|-----|
| Standard RLHF pipeline replacement | DPO | Well-studied, stable, strong baselines |
| Only have thumbs up/down data | KTO | Does not require paired preferences |
| Limited GPU memory | SimPO | No reference model needed |
| Maximum alignment quality (no cost constraint) | RLHF (PPO) | Online exploration finds edge cases |
| Quick experimentation | SimPO | Simplest setup, fewest components |

**3. Surprises and Insights**
- SimPO's lack of a reference model does not obviously hurt performance at this scale.
- KTO, despite seeing "less information" per example (binary vs paired), is competitive.
- Response length distributions vary across methods -- some methods implicitly learn length biases from the data.
- The compute-quality tradeoff is real: SimPO is fastest (no ref model forward pass), DPO is middle, KTO varies.

**4. Limitations of This Comparison**
- Small model: GPT-2/TinyLlama cannot produce truly helpful responses.
- Limited data: 1.5K examples is far below production scale.
- Proxy metrics: We used perplexity-based reward proxy, not a trained reward model or human evaluation.
- Single seed: Results may vary with different random seeds.
- No hyperparameter search: Default hyperparameters may not be optimal for each method.

**For a production comparison**, you would want:
- 7B+ parameter models
- 50K+ preference examples
- Trained reward model for evaluation
- LLM-as-judge (GPT-4/Claude) win rates
- Multiple seeds with confidence intervals
- Hyperparameter sweeps with matched compute

---
## 10. "Why Does This Work?" -- Deep Understanding Prompts

### Why do different methods produce different results on the same data?

Even though DPO, KTO, and SimPO all optimize for "alignment" on the same underlying preference data, they encode different **inductive biases**:

- **DPO** assumes preferences are generated by a Bradley-Terry model (pairwise comparisons). It learns a reward function implicitly by maximizing the gap between chosen and rejected log-probabilities.

- **KTO** uses a Kahneman-Tversky value function, which models **loss aversion** -- the idea that humans weight losses more than gains. This means KTO penalizes bad outputs more aggressively than it rewards good ones.

- **SimPO** uses the average log-probability of the generated sequence as an implicit reward, rather than learning a separate reward signal. This means it favors responses that the model itself finds likely -- a different optimization target.

These different loss landscapes lead to different local optima, even with identical data.

### The offline vs online distinction: why PPO-based RLHF can outperform DPO

A critical distinction that interviewers love to probe:

- **Offline methods** (DPO, KTO, SimPO): Train on a fixed dataset of preferences. The model never sees its own outputs during training.
- **Online methods** (PPO-RLHF): Generate responses during training, get reward signals, and update. The model explores and learns from its own mistakes.

**Why online can be better:**
1. **Distribution shift**: Offline methods train on data from a different policy (the SFT model). As the model changes during training, the training data becomes off-policy.
2. **Exploration**: Online methods discover failure modes and edge cases that fixed datasets miss.
3. **Self-play dynamics**: Online RL creates a curriculum -- the model faces increasingly challenging scenarios.

**Why offline is still popular:**
1. Much simpler to implement (no reward model, no RL infra)
2. More stable training (no reward hacking, no mode collapse from RL)
3. Often 80-90% of the quality at 10% of the complexity

### Data quality vs method choice: which matters more?

**Data quality matters more.** This is perhaps the most important practical insight:

- Zephyr-7B (Tunstall et al. 2023) showed that distilled SFT on UltraChat plus distilled DPO on UltraFeedback beat much larger RLHF-trained models (e.g., Llama-2-Chat-70B) on MT-Bench and AlpacaEval.
- The Llama 2 report emphasizes careful preference-annotation quality control and iterative data collection in its RLHF pipeline; a common reading is that data quality mattered at least as much as the algorithmic choices (this is an interpretation, not a quoted finding).
- Anthropic's Constitutional AI work suggests that the quality of the principles matters more than the specific optimization procedure.

**Interview-ready formulation:** "Spending an extra week on data curation typically yields more improvement than spending an extra week on algorithmic innovation. The method converts data quality into model quality -- the conversion efficiency matters less than the input quality."

---
## Interview Question Bank: Alignment Method Comparison

*This notebook is your portfolio piece. The questions below test whether you can design and execute rigorous experiments -- the core skill for any ML researcher role.*

---

**Q1: "You ran DPO, KTO, and SimPO on the same data and DPO won. Does that mean DPO is the best method?"**

**What we're testing:** Experimental rigor, understanding of confounders, scientific reasoning under ambiguity.

**Good answer:**
- No, a single experiment is not sufficient. Results depend on: hyperparameter tuning (was each method equally tuned?), data characteristics (paired data naturally favors DPO), evaluation methodology (what metric? what judge?), and model scale.
- The comparison is only valid if all methods received the same tuning budget.

**Great answer (Senior -> Principal level):**
- "A fair comparison requires controlling for: (1) compute budget per method (not just epochs, but total FLOPs including reference model overhead), (2) hyperparameter search budget (DPO has fewer hyperparameters than KTO, giving it a tuning advantage), (3) evaluation diversity (single metric comparisons are misleading -- check helpfulness, safety, and diversity independently), and (4) data distribution match (DPO is designed for paired data, so testing on paired data is a home-court advantage)."
- Proposes the right experiment: "Fix total compute, give each method equal hyperparameter search budget, evaluate on multiple axes with both automated judges and human eval, test on 3+ data distributions."
- Notes the meta-insight: "In practice, the method matters less than the data quality, the number of iterative refinement rounds, and the evaluation rigor. Large data-quality improvements routinely outperform any algorithm change."

**Red flag:** "DPO won so DPO is best." No discussion of experimental validity. Takes benchmark results at face value.

**Follow-up:** "Your paper reviewer says your comparison is unfair because DPO had more hyperparameter tuning. How do you respond?"
- Expected: Acknowledge the concern, describe the tuning protocol used, propose a compute-matched comparison, offer to release all configs and sweep results for reproducibility.

---

**Q2: "Design an evaluation protocol that can distinguish between 'better aligned' and 'better at gaming the evaluation.'"**

**What we're testing:** Evaluation sophistication, awareness of Goodhart's Law in alignment evaluation.

**Good answer:**
- Use multiple evaluators: reward model scores, GPT-4/Claude as judge, and human evaluation
- Include adversarial prompts designed to exploit common gaming strategies (e.g., length, hedging, false confidence)
- Check for reward hacking indicators: length inflation, repetitive safety disclaimers, vague but "safe" responses

**Great answer (Senior -> Principal level):**
- "The fundamental problem is that any single metric can be gamed. My protocol:
  1. **Diverse prompt distribution**: Not just helpfulness -- include safety, refusals, multi-turn, instruction-following, edge cases
  2. **Held-out evaluator**: Train the model against one reward model, evaluate with a completely different one. If the model games the training RM but not the held-out RM, you have overfitting.
  3. **Human evaluation with calibration**: Small-scale (200-500 examples) but with calibrated annotators. Include 'trap' questions where you know the right answer.
  4. **Behavioral probes**: Test specific failure modes -- does the model become more sycophantic? More verbose? More hedging? These are measurable without subjective judgment.
  5. **Out-of-distribution generalization**: Test on prompt distributions NOT in the training data. A model that is truly better-aligned should generalize; a model that gamed the metric will not."

**Red flag:** "Just use the reward model score." Single-metric thinking. No awareness of Goodhart's Law.

---

**Q3: "You are writing a research paper comparing alignment methods. What controls do you need?"**

**What we're testing:** Research methodology, publication-readiness, attention to detail.

**Great answer:**
- Same base model (same SFT checkpoint as starting point for all methods)
- Same training data (or equivalent -- KTO needs unpaired, so derive from the same source)
- Compute-matched comparison (total FLOPs, not just epochs)
- Multiple random seeds (at least 3, report mean and standard deviation)
- Hyperparameter search with equal budget per method
- Multiple evaluation metrics (automated + human, multiple prompt distributions)
- Ablation studies: what happens when you change one variable (data size, beta, model scale)?
- Report negative results and failure modes, not just best numbers
- Release all code, data, and configs for reproducibility

---
## Production Implementation Notes: Running Fair Alignment Comparisons

### The Experiment Protocol Used at Frontier Labs

When a post-training team needs to decide between alignment methods, here is the standard protocol:

**Phase 1: Small-scale screening (1-2 days)**
- 1B parameter model, 5K preference examples
- Train each candidate method for a fixed number of steps (not epochs -- step count normalizes for batch size differences)
- Quick evaluation: reward model score + 3-5 manual spot checks
- Goal: eliminate clearly bad methods/configs before spending real compute

**Phase 2: Medium-scale comparison (1 week)**
- 7B parameter model, full preference dataset (50K-200K examples)
- Equal hyperparameter search budget per method (e.g., 20 runs each via Bayesian optimization)
- Evaluation: automated metrics (win rate, safety, diversity) + 100 human comparisons
- Goal: identify top 2-3 configurations

**Phase 3: Full-scale validation (2-4 weeks)**
- Target model size (70B+), full pipeline
- Run top configurations from Phase 2
- Full human evaluation (500+ comparisons), safety red-teaming, capability regression testing
- Goal: final model selection for release

### Key Engineering Details

- **Reproducibility**: Fix all random seeds, log every hyperparameter, checkpoint every N steps. You will need to reproduce results when a reviewer (or your VP) asks.
- **Compute accounting**: Track GPU-hours per method, not just wall-clock time. DPO uses 2 models; SimPO uses 1. A "fair" comparison should account for this.
- **Statistical significance**: With N=50 evaluation examples, differences under 5% win rate are noise. Budget for N=200+ to detect meaningful differences. Use bootstrap confidence intervals.

---
## How Alignment Comparisons Get Tested in Interviews

### The Portfolio Presentation

This notebook is designed to be a portfolio piece. In a research interview, you may be asked to present past work. Here is how to present an alignment comparison:

**The wrong way:** "I trained DPO, KTO, and SimPO and here are the numbers."

**The right way:** "Design a controlled experiment to understand when each alignment method is most effective. Here are the 5 controls I used, here is what I found, and here is why the results surprised me."

### What Interviewers Probe

1. **Experimental design**: "Why did you choose these methods and not others?" -- You should have a principled reason for inclusion/exclusion.
2. **Confounders**: "How do you know the difference is not due to hyperparameter tuning?" -- You should have a tuning protocol and be able to describe it.
3. **Generalization**: "Would these results hold at 70B scale?" -- Honest answer: "Probably directionally, but the margins may change. Here is what I would check."
4. **Actionable conclusions**: "Based on this, what would you recommend for our next model?" -- This is the real test. Can you go from data to decision?

### Presenting Negative Results

Principal-level candidates are comfortable saying: "Method X did not work as well as I expected, and here is my hypothesis for why." This demonstrates intellectual honesty and analytical depth -- both are valued much more than cherry-picked positive results.

---
## 11. Flashcard Summary

| # | Question | Answer |
|---|----------|--------|
| 1 | What are the three key variables to control in a fair alignment comparison? | Same base model, same data, same compute budget |
| 2 | What data format does DPO require vs KTO? | DPO: paired preferences (chosen/rejected for same prompt). KTO: unpaired binary (good/bad independently) |
| 3 | What is SimPO's key simplification over DPO? | SimPO removes the reference model, using average sequence log-probability as implicit reward |
| 4 | Why can PPO-RLHF outperform DPO despite DPO's simplicity? | PPO is online (sees own outputs during training), enabling exploration and avoiding distribution shift |
| 5 | What matters more for alignment quality: method choice or data quality? | Data quality. High-quality data with a simple method beats low-quality data with a sophisticated method |
| 6 | What is the typical memory advantage of SimPO over DPO? | It drops the frozen reference model, which is forward-only (no gradients/optimizer states), so the saving is well under 50% of training memory; the bigger wins are compute and simplicity |
| 7 | What is the Bradley-Terry model and why does it matter for DPO? | A probabilistic model of pairwise comparisons. DPO assumes preferences follow this model, which may not hold for all human preferences |
| 8 | What is "loss aversion" in KTO and why does it help? | From Kahneman-Tversky: humans weight losses more than gains. KTO penalizes bad outputs more than it rewards good ones, matching human psychology |
| 9 | Name 5 metrics for comparing alignment methods | Reward model score, win rate (LLM-as-judge), KL divergence from reference, response length distribution, perplexity |
| 10 | What is the key limitation of using perplexity as a reward proxy? | Perplexity measures fluency/likelihood, not helpfulness or safety. A harmful but fluent response can have low perplexity |

---
## 12. Portfolio Note: How to Present This in an Interview

### Key Talking Points

1. **Lead with the methodology, not the results:** "Design a controlled comparison framework that isolates the effect of the alignment algorithm from confounding factors like data quality and compute budget."

2. **Show you understand limitations:** "This is a proof-of-concept at small scale. For production conclusions, you would need 7B+ models, trained reward models for evaluation, and multiple seeds. The value here is in the framework, not the specific numbers."

3. **Demonstrate practical judgment:** "The key takeaway is that data quality dominates method choice. Given a choice of where to invest engineering effort, the higher-leverage move is to invest in better preference data over a fancier alignment algorithm."

4. **Connect to real deployment:** "In production at a large tech company, I would start with DPO for its stability and simplicity, then consider SimPO if GPU memory is constrained, and only invest in full RLHF if we need the last few percent of quality."

5. **Show research awareness:** "The field is moving toward online DPO variants (like OAIF and online DPO) that combine the simplicity of DPO with the exploration benefits of PPO. I would track those developments."

### Likely Follow-Up Questions

- "How would you scale this comparison to a 70B model?" (LoRA/QLoRA, distributed training, gradient checkpointing)
- "What if the reward model is wrong?" (Reward hacking, Goodhart's law, ensemble reward models)
- "How do you handle multi-objective alignment?" (Safety vs helpfulness tradeoff, Pareto frontiers)
- "What about constitutional AI vs RLHF?" (Rule-based vs learned rewards, scalable oversight)

In [ ]:
# Cleanup
del models, ref_model_eval
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
print("Notebook complete. All models and results saved.")